# Adjacent Chunk Retriever Demo
Below is a working example of using an adjacent chunk retriever for querying a database. It uses a simple PDF scraper as an example, and ingests to Weaviate. The only important difference on ingestion compared to the pipeline is that a `chunk_index` parameter is passed to the metadata, which is unique for each chunk, regardless of source document.

The query then finds the adjacent chunks in the collection, checks that they are from the same source, and appends them to the original chunk. The intent of this change would be to stike a balance between accuracy and quality of context retrieval by using smaller size chunks for similarity searchs and feeding the LLM larger chunks for ample context.

In [ ]:
import os
from dotenv import load_dotenv
import time
import gc
import logging


import weaviate
from weaviate.classes.init import Auth
from weaviate.client import WeaviateClient
from weaviate.classes.query import Filter
from weaviate.config import AdditionalConfig, Timeout

from langchain_openai import OpenAIEmbeddings
from langchain_weaviate.vectorstores import WeaviateVectorStore
from langchain_core.documents.base import Document
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
_log = logging.getLogger(__name__)

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

In [ ]:
def scrape_pdfs(pdf_directory: str) -> list[Document]:
    """Load and scrape PDFs from a directory into a LangChain document
    object.
    """

    pdf_files = [f for f in os.listdir(pdf_directory) if f.endswith(".pdf")]

    if not pdf_files:
        raise FileNotFoundError("No PDF files found in directory")

    documents = []

    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_directory, pdf_file)

        loader = PyMuPDFLoader(pdf_path)
        document = loader.load()
        documents += document

    return documents

In [ ]:
def chunk_docs(docs: list[Document],
               chunk_size: int = 1000,
               chunk_overlap: int = 50,
) -> list[Document]:
    """Chunk langchain documents and add source_id to metadata.
    Parameters
    ----------
        docs : list
            name of the list of langchain documents
        chunk_size : int
            size of chunks (in characters)
        chunk_overlap : int
            overlap of chunks (in characters)
    Returns
    -------
        docs : list
            list of langchain documents
        """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = text_splitter.split_documents(docs)

    return chunks

In [ ]:
def push_docs_to_weaviate(
    client: WeaviateClient,
    index_name: str,
    docs: list[Document],
) -> bool:
    """Push a list of documents from a pickle file into Weaviate.

    Parameters
    ----------
    client: WeaviateClient
        Weaviate client instance.
    index_name: str
        Name of Weaviate collection to ingest to.
    docs : list
        name of the list of LangChain documents.

    Returns
    -------
    bool
        If True, docs successfully pushed to Weaviate.
    """
    try:
        # Add chunk_index to metadata
        for i, chunk in enumerate(docs):
            chunk.metadata["chunk_index"] = i

        embeddings = OpenAIEmbeddings(
            api_key=openai_api_key, # type: ignore[arg-type]
            model="text-embedding-3-small",
            dimensions=1536,
        )

        vectorstore = WeaviateVectorStore(
            embedding=embeddings,
            index_name=index_name,
            client=client,
            text_key="page_content",
        )

        _log.info(f"Pushing {len(docs)} docs to Weaviate.")
        start_time = time.time()

        vectorstore.add_documents(
            documents=docs,
            embedding=embeddings,
            index_name=index_name,
            client=client,
            text_key="page_content",
            attributes=list(docs[0].metadata.keys())
            if docs[0].metadata
            else [],
        )

        del docs
        gc.collect()

    except Exception as e:
        _log.error(f"Failed to push docs: {e}")
        return False

    else:
        end_time = time.time()
        _log.info(f"Done in {end_time - start_time:.2f}s")
        return True

In [ ]:
def query_adjacent_chunks(client: WeaviateClient,
                          index_name: str,
                          query: str,
                          k: int = 1,
                          adjacent_k: int = 1
                          ) -> list[Document]:
    """Use chunk_index in document metadata to query adjacent chunks.

    Parameters
    ----------
    client: WeaviateClient
        Connection to the Weaviate client.
    index_name: str
        Name of Weaviate collection to ingest to.
    query: str
        The question from the user.
    k: int = 1
        k-value, or the number of chunks to retrieve.
    adjacent_k: int = 1
        number of chunks to pull from either side of each retrieved chunk.
        For example, if adjacent_k was 1, one chunk would be appended to
        either side of the retrieved chunk, for a total of 3 chunks.
    
    Returns
    -------
    """
    # Instantiate vector store
    embeddings = OpenAIEmbeddings(
        api_key=openai_api_key, # type: ignore[arg-type]
        model="text-embedding-3-small",
        dimensions=1536,
    )

    vectorstore = WeaviateVectorStore(
        embedding=embeddings,
        index_name=index_name,
        client=client, 
        text_key="page_content",
    )
    
    # Ensure k is positive
    if adjacent_k < 0:
        raise ValueError("adjacent_k must be a non-negative integer")

    # Retrieve the best match
    top_chunks = vectorstore.similarity_search(query, k=k)

    results = []

    for top_chunk in top_chunks:
        # Extract chunk_index and source for filtering adjacent chunks
        chunk_index = top_chunk.metadata.get("chunk_index")
        source = top_chunk.metadata.get("source")

        lower_bound = chunk_index - adjacent_k
        upper_bound = chunk_index + adjacent_k

        filters = (
            Filter.by_property("chunk_index").greater_or_equal(lower_bound) &
            Filter.by_property("chunk_index").less_or_equal(upper_bound) &
            Filter.by_property("source").equal(source)
        )

        all_chunks = vectorstore.similarity_search(
            query="",  # Empty because we use filters
            filters=filters,
            )
        all_chunks.sort(key=lambda x: x.metadata["chunk_index"])
        print("Chunk indices:", [chunk.metadata["chunk_index"] for chunk in all_chunks])
        combined_text = "\n\n".join(chunk.page_content for chunk in all_chunks)

        document = Document(
            metadata=top_chunk.metadata,  # Use matching chunk metadata
            page_content=combined_text
        )
        results.append(document)

    return results

### ***Make sure to only add documents to the database once***

In [ ]:
try:
    client = weaviate.connect_to_custom(
            http_host=http_host,
            http_port=8080,  # Default is 80, WCD uses 443
            http_secure=False,
            grpc_host=grpc_host,
            grpc_port=50051,  # Default is 50051, WCD uses 443
            grpc_secure=False,
            auth_credentials=Auth.api_key(
                weaviate_api_key
            ),  # The API key to use for authentication
            headers={"X-OpenAI-Api-Key": openai_api_key},
            additional_config=AdditionalConfig(
                timeout=Timeout(init=30, query=400, insert=400)  # Values in seconds
            )
        )
    print("Client is live:", client.is_ready())

    # # Add documents to the database, only run this once
    # scraped_docs = scrape_pdfs("./pdfs")
    # docs = chunk_docs(scraped_docs, chunk_size=500, chunk_overlap=0)
    # vectorstore = push_docs_to_weaviate(client, "Adjacent_Chunks_Demo", docs)

    # Query the database
    results = query_adjacent_chunks(
        client,
        "Adjacent_Chunks_Demo",
        "What are the 4 main science drivers?",
        k=2,
        adjacent_k=2
    )

    for i, doc in enumerate(results, 1):
        print(f"\n--- Document {i} ---")
        print("Source:", doc.metadata.get("source"))
        print("Content:\n", doc.page_content)

except Exception as e:
    print("Error:", e)
finally:
    client.close()